# ДЗ 2.2 Задание 3

для каждого пользователя из ids.csv получить шесть числовых признаков - минимум и максимум watched_time для фильмов с оценками 8, 9, 10. Все пользователи выборки должны остаться, даже если подходящих данных нет.

## 1. Подключение Spark и чтение CSV

ids.csv - список пользователей. cut_transactions.csv содержит продолжительности просмотров, cut_ratings.cs - оценки фильмов. Для соединения используем пару (user_uid, element_uid)

In [1]:
import pandas as pd
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, LongType, IntegerType, DoubleType
from hw22_utils import BASE_DIR, S3_ROOT, ensure_data, configure_s3

spark = (SparkSession.builder.appName('DWH_HW2_2_Task3_Features')
         .master('local[4]')
         .config('spark.sql.session.timeZone', 'UTC')
         .getOrCreate())
configure_s3(spark)
assert spark.version.startswith('3.5.')

folder = ensure_data('task3')
ids_schema = StructType([StructField('user_uid', LongType(), True)])
tx_schema = StructType([
    StructField('element_uid', LongType(), True),
    StructField('user_uid', LongType(), True),
    StructField('watched_time', LongType(), True),
])
ratings_schema = StructType([
    StructField('user_uid', LongType(), True),
    StructField('element_uid', LongType(), True),
    StructField('rating', IntegerType(), True),
    StructField('ts', DoubleType(), True),
])

ids = spark.read.option('header', True).schema(ids_schema).csv(str(folder / 'ids.csv'))
views = (spark.read.option('header', True).schema(tx_schema)
         .csv(str(folder / 'cut_transactions.csv')))
ratings = (spark.read.option('header', True).schema(ratings_schema)
           .csv(str(folder / 'cut_ratings.csv')))
print('Пользователей выборки:', ids.count())
print('Просмотров:', views.count())
print('Оценок:', ratings.count())
ids.show(5)
views.show(5)
ratings.show(5)

Пользователей выборки: 100
Просмотров: 274385
Оценок: 438790
+--------+
|user_uid|
+--------+
|  110138|
|  412991|
|   83691|
|  334052|
|   24124|
+--------+
only showing top 5 rows

+-----------+--------+------------+
|element_uid|user_uid|watched_time|
+-----------+--------+------------+
|       6130|  563180|        3264|
|       2771|  133315|       10557|
|        984|  290104|        6845|
|       6980|  385385|        2734|
|       2714|  408794|        7583|
+-----------+--------+------------+
only showing top 5 rows

+--------+-----------+------+--------------------+
|user_uid|element_uid|rating|                  ts|
+--------+-----------+------+--------------------+
|  571252|       1364|    10| 4.430517426309871E7|
|   63140|       3037|    10|4.4305139282818206E7|
|  443817|       4363|     8| 4.430513620584908E7|
|  359870|       1364|    10| 4.430506300637321E7|
|  359870|       3578|     9| 4.430506073913281E7|
+--------+-----------+------+--------------------+
only sh

## 2. Проверка ключей и повторяющихся строк

Если одна пара (user_uid, element_uid) встречается несколько раз в обоих источниках, обычное соединение может размножить строки. Для приложенного набора повторяющихся пар нет.

In [2]:
ids_duplicates = ids.groupBy('user_uid').count().filter('count > 1').count()
view_pair_duplicates = (views.groupBy('user_uid', 'element_uid').count()
                        .filter('count > 1').count())
rating_pair_duplicates = (ratings.groupBy('user_uid', 'element_uid').count()
                          .filter('count > 1').count())
missing_keys = {
    'ids': ids.filter('user_uid IS NULL').count(),
    'views': views.filter('user_uid IS NULL OR element_uid IS NULL').count(),
    'ratings': ratings.filter('user_uid IS NULL OR element_uid IS NULL').count(),
}
print('Повторы user_uid в выборке:', ids_duplicates)
print('Повторы (user_uid, element_uid) в просмотрах:', view_pair_duplicates)
print('Повторы (user_uid, element_uid) в оценках:', rating_pair_duplicates)
print('Пустые ключи:', missing_keys)
assert ids_duplicates == view_pair_duplicates == rating_pair_duplicates == 0
assert all(x == 0 for x in missing_keys.values())
assert views.filter('watched_time IS NULL OR watched_time < 0').count() == 0
print('Соединение по паре ключей не создаст случайных повторов.')

Повторы user_uid в выборке: 0
Повторы (user_uid, element_uid) в просмотрах: 0
Повторы (user_uid, element_uid) в оценках: 0
Пустые ключи: {'ids': 0, 'views': 0, 'ratings': 0}
Соединение по паре ключей не создаст случайных повторов.


## 3. Отбор нужных пользователей и оценок

Сначала отбираем оценки 8, 9 и 10. Затем ограничиваем оба крупных источника пользователями из ids.csv. 

 Фильм учитывается тогда, когда для того же пользователя и того же фильма есть и просмотр, и оценка.

In [3]:
target_users = ids.select('user_uid').distinct()
views_small = views.join(F.broadcast(target_users), on='user_uid', how='inner')
ratings_small = (ratings.filter(F.col('rating').isin(8, 9, 10))
                 .join(F.broadcast(target_users), on='user_uid', how='inner'))
matched = (views_small.join(ratings_small.select('user_uid', 'element_uid', 'rating'),
                            on=['user_uid', 'element_uid'], how='inner')
           .select('user_uid', 'element_uid', 'watched_time', 'rating'))
print('Просмотров с оценками 8–10 у пользователей выборки:', matched.count())
matched.orderBy('user_uid', 'element_uid').show(10, truncate=False)

Просмотров с оценками 8–10 у пользователей выборки: 150
+--------+-----------+------------+------+
|user_uid|element_uid|watched_time|rating|
+--------+-----------+------------+------+
|24124   |747        |1575        |9     |
|24124   |7026       |16          |8     |
|24124   |7292       |2578        |9     |
|28855   |747        |7500        |10    |
|28855   |7371       |3887        |9     |
|43752   |823        |9115        |8     |
|43752   |5129       |3437        |10    |
|55511   |1375       |1293        |9     |
|74940   |96         |5683        |10    |
|74940   |747        |15330       |10    |
+--------+-----------+------------+------+
only showing top 10 rows



## 4. Расчёт признаков

Для каждого пользователя применяем условные функции min и max, так значение watched_time попадает в расчёт только для нужной оценки. Если просмотр с определённой оценкой отсутствует, when возвращает NULL, а min и max остаются NULL.

In [4]:
aggregations = []
for rating in (8, 9, 10):
    value = F.when(F.col('rating') == rating, F.col('watched_time'))
    aggregations.append(F.min(value).alias(f'feat_min_watched_time_rating_{rating}'))
    aggregations.append(F.max(value).alias(f'feat_max_watched_time_rating_{rating}'))

per_user = matched.groupBy('user_uid').agg(*aggregations)
features = target_users.join(per_user, on='user_uid', how='left')
COLUMN_ORDER = ['user_uid'] + [
    f'feat_{func}_watched_time_rating_{rating}'
    for rating in (8, 9, 10) for func in ('min', 'max')
]
features = features.select(*COLUMN_ORDER)
features.orderBy('user_uid').show(20, truncate=False)
print('Количество пользователей в результате:', features.count())

+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+
|8504    |NULL                          |NULL                          |NULL                          |NULL                          |NULL                           |NULL                           |
|24124   |16                            |16                            |1575                          |2578                          |NULL                           |NULL                           |
|2525

## 5. Проверки корректности признаков

Проверяем требования: 

(1) одна строка на пользователя; 

(2) все пользователи выборки сохранены; 

(3) значение минимума не превосходит максимум;

(4) отсутствующие оценки дают две пустые колонки; 

(5) встречаются примеры пользователей с одной и несколькими оценками и без подходящих данных.

In [5]:
num_ids = target_users.count()
num_rows = features.count()
num_distinct = features.select('user_uid').distinct().count()
missing_users = target_users.exceptAll(features.select('user_uid')).count()
extra_users = features.select('user_uid').exceptAll(target_users).count()
assert num_ids == num_rows == num_distinct == 100
assert missing_users == extra_users == 0

for r in (8, 9, 10):
    a = f'feat_min_watched_time_rating_{r}'
    b = f'feat_max_watched_time_rating_{r}'
    assert features.filter(F.col(a).isNull() != F.col(b).isNull()).count() == 0
    assert features.filter(F.col(a) > F.col(b)).count() == 0

present_ratings = F.lit(0)
for r in (8, 9, 10):
    present_ratings = present_ratings + F.when(
        F.col(f'feat_min_watched_time_rating_{r}').isNotNull(), F.lit(1)
    ).otherwise(F.lit(0))
examples = features.withColumn('rating_groups_present', present_ratings)
distribution = examples.groupBy('rating_groups_present').count().orderBy('rating_groups_present')
distribution.show()
for count in (0, 1, 2, 3):
    print(f'Пример пользователя с {count} подходящими оценками:')
    examples.filter(F.col('rating_groups_present') == count).show(1, truncate=False)

print('Сохранена вся выборка:', num_rows, 'пользователей.')
print('Все проверки минимумов, максимумов и NULL пройдены.')

+---------------------+-----+
|rating_groups_present|count|
+---------------------+-----+
|                    0|   41|
|                    1|   35|
|                    2|   19|
|                    3|    5|
+---------------------+-----+

Пример пользователя с 0 подходящими оценками:
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+---------------------+
|user_uid|feat_min_watched_time_rating_8|feat_max_watched_time_rating_8|feat_min_watched_time_rating_9|feat_max_watched_time_rating_9|feat_min_watched_time_rating_10|feat_max_watched_time_rating_10|rating_groups_present|
+--------+------------------------------+------------------------------+------------------------------+------------------------------+-------------------------------+-------------------------------+---------------------+
|296003  |NULL                          |NULL     

## 6. Независимая проверка по исходным CSV
 Сравним каждую строку и все шесть признаков с независимым результатом. 

Для исходного архива контрольные значения: 100 пользователей, у 59 есть подходящие просмотры, у 41 все шесть признаков пустые.

In [6]:
ref_path = BASE_DIR / 'task3_reference.csv'
if ref_path.exists():
    reference = spark.read.option('header', True).option('inferSchema', True).csv(str(ref_path))
    for column in COLUMN_ORDER:
        reference = reference.withColumn(column, F.col(column).cast('long'))
    reference = reference.select(*COLUMN_ORDER)
    difference_forward = features.exceptAll(reference).count()
    difference_backward = reference.exceptAll(features).count()
    print('Есть в Spark, нет в контрольном файле:', difference_forward)
    print('Есть в контрольном файле, нет в Spark:', difference_backward)
    assert difference_forward == difference_backward == 0
    print('Полное совпадение с независимым расчётом подтверждено.')
else:
    print('Контрольный CSV не найден; основные проверки из раздела 5 остаются действующими.')

all_empty = features.filter(
    F.col('feat_min_watched_time_rating_8').isNull() &
    F.col('feat_min_watched_time_rating_9').isNull() &
    F.col('feat_min_watched_time_rating_10').isNull()
).count()
assert all_empty == 41
assert num_rows - all_empty == 59
print('Без подходящих просмотров:', all_empty)
print('С подходящими просмотрами:', num_rows - all_empty)

Есть в Spark, нет в контрольном файле: 0
Есть в контрольном файле, нет в Spark: 0
Полное совпадение с независимым расчётом подтверждено.
Без подходящих просмотров: 41
С подходящими просмотрами: 59


## 7. Сохранение результата

Сохраним таблицу в Parquet в новую папку RustFS. Дополнительно запишем CSV рядом с ноутбуком.

In [7]:
target = f'{S3_ROOT}/task3/features/'
features.write.mode('overwrite').parquet(target)
confirmed = spark.read.parquet(target).select(*COLUMN_ORDER)
assert features.exceptAll(confirmed).count() == 0
assert confirmed.exceptAll(features).count() == 0

out_dir = BASE_DIR / 'results'
out_dir.mkdir(exist_ok=True)
features.orderBy('user_uid').toPandas().to_csv(
    out_dir / 'task3_features.csv', index=False, na_rep=''
)
print('S3:', target)
print('Локальный результат:', out_dir / 'task3_features.csv')
print('Все 100 пользователей и все шесть признаков сохранены.')

S3: s3a://dwh-homework/hw22/task3/features/
Локальный результат: /home/jovyan/work/hw2_2/results/task3_features.csv
Все 100 пользователей и все шесть признаков сохранены.


## 8. Выводы

Для 100 пользователей из выборки рассчитаны шесть признаков: минимальное и максимальное значения watched_time отдельно для оценок 8, 9 и 10.

Просмотры и оценки соединены по паре (user_uid, element_uid) после отбора оценок 8–10 и пользователей из выборки. Во входных данных нет повторяющихся пар ключей, поэтому дополнительные правила объединения повторных просмотров не потребовались. Получено 150 подходящих пар «просмотр–оценка», watched_time использован в исходных единицах.

В результате сохранены все 100 пользователей, по одной строке на человека. У 59 есть хотя бы один подходящий просмотр; у 41 все шесть признаков равны NULL. Если отсутствуют просмотры с определённой оценкой, оба признака для неё также равны NULL. Для непустых пар во всех случаях выполнено минимум ≤ максимум. Проверены примеры пользователей без подходящих оценок, с одной, двумя и тремя оценками.

Контрольное сравнение по всем шести признакам показало 0 различий. Итоговая таблица сохранена в Parquet в S3 и в results/task3_features.csv.